# 04 · Gas prices and the generation mix
**Question:** when natural gas gets expensive, how much does ERCOT shift from gas to coal?

Gas and coal compete for the load left after wind, solar and nuclear, so the outcome is
**gas share of fossil generation** = gas / (gas + coal), by day. Price is the daily Henry Hub spot
(weekends and holidays carry the last trading day's price via an ASOF join).
Uri (Feb 10–20 2021) is excluded: prices spiked to $23.86 while plants were failing, which is not economic dispatch.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
con = duckdb.connect("data/grid.duckdb", read_only=True)
con.execute("SET TimeZone = 'UTC'")
q = lambda sql: con.execute(sql).df()

In [2]:
q("SELECT * FROM gas_price_quintiles")

,price_quintile,avg_price_usd,gas_fossil_share_pct,gas_share_pct,coal_share_pct,avg_net_load_mw,days
0,1,2.89,72.1,42.5,15.9,33131.0,550
1,2,3.10,71.9,42.7,16.1,33626.0,502
2,3,3.27,71.8,42.7,16.3,33715.0,501
3,4,3.44,71.9,43.5,16.5,34655.0,497
4,5,3.80,71.6,43.4,16.8,34697.0,494


In [3]:
q("SELECT * FROM gas_price_by_year")

,year,avg_price_usd,q1_price,q5_price,q1_gas_fossil_pct,q5_gas_fossil_pct,shift_pts
0,2019,2.55,2.39,2.74,68.4,67.8,-0.6
1,2020,2.02,1.79,2.25,71.9,70.2,-1.7
2,2021,3.73,3.46,4.06,66.6,67.8,1.2
3,2022,6.43,5.39,7.47,69.4,70.4,0.9
4,2023,2.53,2.30,2.77,75.4,75.0,-0.4
5,2024,2.25,1.85,3.08,78.1,76.2,-1.9
6,2025,3.53,3.08,4.26,75.0,74.0,-1.0


## Controlled regression
Raw comparisons are confounded: gas prices run high in cold snaps and heat waves, exactly when coal is already running flat out.
So regress gas share of fossil on log price, controlling for net load (and its square), month and year.
Standard errors are Newey-West (HAC, 7 lags) because neighbouring days are correlated.

In [4]:
import statsmodels.formula.api as smf
d = q("SELECT * FROM gas_daily WHERE NOT is_uri")
d["log_p"] = np.log(d.henry_hub_usd)
d["nl_gw"] = d.net_load_mwh / 24 / 1000
LN2 = np.log(2)
pooled = smf.ols("gas_fossil_share_pct ~ log_p + nl_gw + I(nl_gw**2) + C(month) + C(year)", d).fit(
    cov_type="HAC", cov_kwds={"maxlags": 7})
b, se = pooled.params["log_p"] * LN2, pooled.bse["log_p"] * LN2
print(f"Doubling Henry Hub shifts gas share of fossil generation by {b:+.2f} pts "
      f"(95% CI {b - 1.96 * se:+.2f} to {b + 1.96 * se:+.2f}); R^2 {pooled.rsquared:.2f}")
print(f"Average gas share of fossil generation: {d.gas_fossil_share_pct.mean():.1f}%")

Doubling Henry Hub shifts gas share of fossil generation by -1.55 pts (95% CI -2.27 to -0.83); R^2 0.68
Average gas share of fossil generation: 71.9%


In [5]:
rows = []
for yr, g in d.groupby("year"):
    m = smf.ols("gas_fossil_share_pct ~ log_p + nl_gw + I(nl_gw**2) + C(month)", g).fit(
        cov_type="HAC", cov_kwds={"maxlags": 7})
    rows.append({"year": yr, "shift_per_doubling_pts": m.params["log_p"] * LN2, "ci95": 1.96 * m.bse["log_p"] * LN2,
                 "price_min": g.henry_hub_usd.min(), "price_max": g.henry_hub_usd.max(),
                 "gas_fossil_share_pct": g.gas_fossil_share_pct.mean(), "coal_share_pct": g.coal_share_pct.mean(),
                 "days": len(g)})
by_year = pd.DataFrame(rows).round(2)
by_year.to_csv("data/clean/gas_switching_by_year.csv", index=False)
by_year

,year,shift_per_doubling_pts,ci95,price_min,price_max,gas_fossil_share_pct,coal_share_pct,days
0,2019,-8.74,4.10,1.75,4.25,68.73,20.54,365
1,2020,-4.84,1.56,1.33,3.14,70.69,18.13,366
2,2021,-3.84,7.99,2.39,6.37,66.73,19.15,354
3,2022,-2.70,1.59,3.46,9.85,69.67,16.92,365
4,2023,-0.92,2.34,1.74,3.78,75.43,13.89,365
5,2024,-1.97,1.76,1.21,13.20,76.91,12.64,366
6,2025,-2.88,2.21,2.65,9.86,74.82,12.94,363


In [6]:
import plotly.graph_objects as go
from viz import add_caption, save, SERIES, INK_2
fig = go.Figure(go.Bar(x=by_year.year, y=by_year.shift_per_doubling_pts, marker_color=SERIES[0], name="Shift per doubling",
                       error_y=dict(type="data", array=by_year.ci95, color=INK_2, thickness=1.5, width=6),
                       hovertemplate="%{x}: %{y:+.1f} pts<extra></extra>"))
fig.add_hline(y=0, line=dict(color=INK_2, width=1))
fig.update_layout(title="How much gas loses to coal when gas prices double, by year", bargap=0.45,
                  xaxis_title="Year", yaxis_title="Change in gas share of fossil generation (pts)", xaxis=dict(dtick=1))
first, late = by_year.iloc[0], by_year[by_year.year >= 2022]
add_caption(fig, f"Fuel switching has faded: {first.shift_per_doubling_pts:+.1f} pts per doubling in {int(first.year)}, {late.shift_per_doubling_pts.min():+.1f} to {late.shift_per_doubling_pts.max():+.1f} since 2022 as coal shrank. Pooled: {b:+.1f} pts. Bars: 95% CI.")
save(fig, "gas_price_switching")